# Eindhoven bicycle accidents filter and year grouping


In [22]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely import wkt

In [23]:
# Set input and output paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "ongevallen_2022_2024.csv"
OUTPUT_PATH = CLEAN_DIR / "ds2.csv"


# Load the raw accident dataset
df = pd.read_csv(INPUT_PATH, sep=",", low_memory=False)

print("Raw dataset shape:", df.shape)
print("Raw columns:")
print(df.columns.tolist())

# Check whether the required columns exist
required_columns = [
    "gemeente",
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
    "shape",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Filter records for cycling-related accidents in Eindhoven
bike_columns = [
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
]

location_mask = (
    df["gemeente"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("eindhoven")
)

bike_mask = False

for col in bike_columns:
    bike_mask = bike_mask | df[col].astype(str).str.contains(
        "fiets|bicycle|bike",
        case=False,
        na=False,
        regex=True,
    )

df_accidents = df[location_mask & bike_mask].copy()

print("Filtered Eindhoven cycling accident records:", len(df_accidents))

Raw dataset shape: (382421, 40)
Raw columns:
['FID', 'verkeersongeval_nummer', 'jaar_ongeval', 'verkeersongeval_afloop', 'aantal_partijen', 'partij_1_objecttype', 'partij_1_objecttype_overig', 'partij_2_objecttype', 'partij_2_objecttype_overig', 'aard_ongeval', 'niveau_koppelen', 'wegsituatie', 'bebouwde_kom', 'maximum_snelheid', 'wegverlichting', 'wegverharding', 'wegdek', 'lichtgesteldheid', 'zichtafstand', 'weersgesteldheid', 'bijz_verkeersmaatregel', 'bijz_verkeersmaatregel_overig', 'bijz_infrastructuur', 'bijz_infrastructuur_overig', 'bijz_tijdelijke_aard', 'bijz_tijdelijke_aard_overig', 'junctie_id', 'wegvak_id', 'hectometer', 'straatnaam', 'woonplaats', 'actueel', 'wegbeheerder', 'gemeente', 'provincie', 'dienstnaam', 'districtnaam', 'indicatie_alcohol', 'gdb_geomattr_data', 'shape']
Filtered Eindhoven cycling accident records: 1653


In [24]:
# 5. Rename Dutch column names to English column names
rename_map = {
    "FID": "fid",
    "verkeersongeval_nummer": "accident_id",
    "jaar_ongeval": "accident_year",
    "verkeersongeval_afloop": "accident_outcome",
    "aantal_partijen": "number_of_parties",
    "partij_1_objecttype": "party_1_object_type",
    "partij_1_objecttype_overig": "party_1_object_type_other",
    "partij_2_objecttype": "party_2_object_type",
    "partij_2_objecttype_overig": "party_2_object_type_other",
    "aard_ongeval": "accident_nature",
    "niveau_koppelen": "linking_level",
    "wegsituatie": "road_situation",
    "bebouwde_kom": "built_up_area",
    "maximum_snelheid": "speed_limit",
    "wegverlichting": "road_lighting",
    "wegverharding": "road_pavement",
    "wegdek": "road_surface",
    "lichtgesteldheid": "light_condition",
    "zichtafstand": "visibility_distance",
    "weersgesteldheid": "weather_condition",
    "bijz_verkeersmaatregel": "special_traffic_measure",
    "bijz_verkeersmaatregel_overig": "special_traffic_measure_other",
    "bijz_infrastructuur": "special_infrastructure",
    "bijz_infrastructuur_overig": "special_infrastructure_other",
    "bijz_tijdelijke_aard": "special_temporary_nature",
    "bijz_tijdelijke_aard_overig": "special_temporary_nature_other",
    "junctie_id": "junction_id",
    "wegvak_id": "road_section_id",
    "hectometer": "hectometer",
    "straatnaam": "street_name",
    "woonplaats": "place_name",
    "actueel": "current",
    "wegbeheerder": "road_authority",
    "gemeente": "municipality",
    "provincie": "province",
    "dienstnaam": "service_area_name",
    "districtnaam": "district_name",
    "indicatie_alcohol": "alcohol_indication",
    "gdb_geomattr_data": "gdb_geometry_attribute_data",
    "shape": "geometry",
}

existing_rename_map = {
    old_name: new_name
    for old_name, new_name in rename_map.items()
    if old_name in df_accidents.columns
}

df_accidents = df_accidents.rename(columns=existing_rename_map)

print("Renamed columns:")
print(df_accidents.columns.tolist())



# Translate selected categorical values to English
if "accident_outcome" in df_accidents.columns:
    print("Raw accident outcome values:")
    print(df_accidents["accident_outcome"].value_counts(dropna=False))

    accident_outcome_map = {
        "Uitsluitend materiele schade": "property_damage_only",
        "Uitsluitend materiële schade": "property_damage_only",
        "Letsel": "injury",
        "Dodelijk": "fatal",
    }

    df_accidents["accident_outcome"] = (
        df_accidents["accident_outcome"]
        .map(accident_outcome_map)
        .fillna(df_accidents["accident_outcome"])
    )

    print("Translated accident outcome values:")
    print(df_accidents["accident_outcome"].value_counts(dropna=False))


# Translate road authority values
if "road_authority" in df_accidents.columns:
    print("Raw road authority values:")
    print(df_accidents["road_authority"].value_counts(dropna=False))

    road_authority_map = {
        "Gemeente": "municipality",
        "Provincie": "province",
        "Rijk": "national_government",
        "Waterschap": "water_authority",
        "Overig": "other",
        "Onbekend": "unknown",
    }

    df_accidents["road_authority"] = (
        df_accidents["road_authority"]
        .map(road_authority_map)
        .fillna(df_accidents["road_authority"])
    )

    print("Translated road authority values:")
    print(df_accidents["road_authority"].value_counts(dropna=False))


# Translate party object type values
object_type_map = {
    "Fiets": "bicycle",
    "e-bike": "e-bike",
    "Personenauto": "passenger_car",
    "Bromfiets": "moped",
    "Snorfiets": "light_moped",
    "Motorfiets": "motorcycle",
    "Vrachtauto": "truck",
    "Bestelauto": "van",
    "Bus": "bus",
    "Voetganger": "pedestrian",
    "Overig": "other",
    "Onbekend": "unknown",
}

object_type_columns = [
    "party_1_object_type",
    "party_1_object_type_other",
    "party_2_object_type",
    "party_2_object_type_other",
]

for col in object_type_columns:
    if col in df_accidents.columns:
        print(f"Raw values in {col}:")
        print(df_accidents[col].value_counts(dropna=False))

        df_accidents[col] = (
            df_accidents[col]
            .map(object_type_map)
            .fillna(df_accidents[col])
        )

        print(f"Translated values in {col}:")
        print(df_accidents[col].value_counts(dropna=False))

# Convert the current-status field from Ja/Nee to boolean values
if "current" in df_accidents.columns:
    current_map = {
        "Ja": True,
        "Nee": False,
        "ja": True,
        "nee": False,
    }

    df_accidents["current"] = (
        df_accidents["current"]
        .map(current_map)
        .fillna(df_accidents["current"])
    )

    print("Converted current values:")
    print(df_accidents["current"].value_counts(dropna=False))

# Validate geometry values
def parse_wkt_geometry(value):
    if pd.isna(value):
        return None

    try:
        return wkt.loads(str(value))
    except Exception:
        return None


if "geometry" in df_accidents.columns:
    geometry_object = df_accidents["geometry"].apply(parse_wkt_geometry)
    invalid_geometry_count = geometry_object.isna().sum()

    print("Invalid or missing geometry rows:", invalid_geometry_count)

Renamed columns:
['fid', 'accident_id', 'accident_year', 'accident_outcome', 'number_of_parties', 'party_1_object_type', 'party_1_object_type_other', 'party_2_object_type', 'party_2_object_type_other', 'accident_nature', 'linking_level', 'road_situation', 'built_up_area', 'speed_limit', 'road_lighting', 'road_pavement', 'road_surface', 'light_condition', 'visibility_distance', 'weather_condition', 'special_traffic_measure', 'special_traffic_measure_other', 'special_infrastructure', 'special_infrastructure_other', 'special_temporary_nature', 'special_temporary_nature_other', 'junction_id', 'road_section_id', 'hectometer', 'street_name', 'place_name', 'current', 'road_authority', 'municipality', 'province', 'service_area_name', 'district_name', 'alcohol_indication', 'gdb_geometry_attribute_data', 'geometry']
Raw accident outcome values:
accident_outcome
Uitsluitend materiele schade    1301
Letsel                           348
Dodelijk                           4
Name: count, dtype: int64

In [25]:
# clean columns
clean_columns = [
    "fid",
    "accident_id",
    "accident_year",
    "accident_outcome",
    "number_of_parties",
    "party_1_object_type",
    "party_1_object_type_other",
    "party_2_object_type",
    "party_2_object_type_other",
    "accident_nature",
    "linking_level",
    "road_situation",
    "built_up_area",
    "speed_limit",
    "road_lighting",
    "road_pavement",
    "road_surface",
    "light_condition",
    "visibility_distance",
    "weather_condition",
    "special_traffic_measure",
    "special_traffic_measure_other",
    "special_infrastructure",
    "special_infrastructure_other",
    "special_temporary_nature",
    "special_temporary_nature_other",
    "junction_id",
    "road_section_id",
    "hectometer",
    "street_name",
    "place_name",
    "current",
    "road_authority",
    "municipality",
    "province",
    "service_area_name",
    "district_name",
    "alcohol_indication",
    "gdb_geometry_attribute_data",
    "geometry",
]

existing_clean_columns = [
    col for col in clean_columns
    if col in df_accidents.columns
]

df_accidents_clean = df_accidents[existing_clean_columns].copy()

print("Final clean dataset shape:", df_accidents_clean.shape)
print("Final clean columns:")
print(df_accidents_clean.columns.tolist())

df_accidents_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
df_accidents_clean.head(7)

Final clean dataset shape: (1653, 40)
Final clean columns:
['fid', 'accident_id', 'accident_year', 'accident_outcome', 'number_of_parties', 'party_1_object_type', 'party_1_object_type_other', 'party_2_object_type', 'party_2_object_type_other', 'accident_nature', 'linking_level', 'road_situation', 'built_up_area', 'speed_limit', 'road_lighting', 'road_pavement', 'road_surface', 'light_condition', 'visibility_distance', 'weather_condition', 'special_traffic_measure', 'special_traffic_measure_other', 'special_infrastructure', 'special_infrastructure_other', 'special_temporary_nature', 'special_temporary_nature_other', 'junction_id', 'road_section_id', 'hectometer', 'street_name', 'place_name', 'current', 'road_authority', 'municipality', 'province', 'service_area_name', 'district_name', 'alcohol_indication', 'gdb_geometry_attribute_data', 'geometry']
Cleaned dataset saved to: /Users/linda/Desktop/2AMD20-Knowledge-Engineering/data/clean/ds2.csv


,fid,accident_id,accident_year,accident_outcome,number_of_parties,party_1_object_type,party_1_object_type_other,party_2_object_type,party_2_object_type_other,accident_nature,...,place_name,current,road_authority,municipality,province,service_area_name,district_name,alcohol_indication,gdb_geometry_attribute_data,geometry
277,ongevallen_2022_2024.278,20220067354,2022,property_damage_only,2,bicycle,NaN,bicycle,NaN,Onbekend,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (155647.4459999986 383051.53500000015)
708,ongevallen_2022_2024.708,20230008779,2023,property_damage_only,2,passenger_car,NaN,moped,NaN,Flank,...,Eindhoven,False,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (160926.5353000015 382865.3002999984)
968,ongevallen_2022_2024.968,20230007858,2023,property_damage_only,2,light_moped,NaN,e-bike,NaN,Flank,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (161137.62170000002 383560.53559999913)
970,ongevallen_2022_2024.970,20230009169,2023,property_damage_only,2,passenger_car,NaN,bicycle,NaN,Onbekend,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (163854.94700000063 383661.2509999983)
1383,ongevallen_2022_2024.1383,20240034437,2024,property_damage_only,2,passenger_car,NaN,bicycle,NaN,Flank,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (161535.47239999846 383981.58729999885)
1450,ongevallen_2022_2024.1451,20240034572,2024,property_damage_only,2,passenger_car,NaN,bicycle,NaN,Onbekend,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (162106 384849)
1480,ongevallen_2022_2024.1481,20240034790,2024,property_damage_only,2,passenger_car,NaN,bicycle,NaN,Flank,...,Eindhoven,True,municipality,Eindhoven,Noord-Brabant,NaN,NaN,NaN,NaN,POINT (159600.72300000116 383551.41899999976)


## accidents in 2024

In [26]:
# Count accident records by year
yearly_counts = df_accidents_clean["accident_year"].value_counts().sort_index()
print("Accident records by year:")
print(yearly_counts)

Accident records by year:
accident_year
2022    545
2023    551
2024    557
Name: count, dtype: int64


In [27]:
# Create and save 2024 preliminary-task dataset
df_accidents_2024 = df_accidents_clean[
    df_accidents_clean["accident_year"] == 2024
].copy()

print("2024 preliminary-task dataset shape:", df_accidents_2024.shape)
print("2024 accident records:", len(df_accidents_2024))


# Save 2024 dataset for preliminary task
PRELIMINARY_2024_PATH = CLEAN_DIR / "ds2_2024.csv"

df_accidents_2024.to_csv(PRELIMINARY_2024_PATH, index=False)

print(f"2024 preliminary-task dataset saved to: {PRELIMINARY_2024_PATH}")

2024 preliminary-task dataset shape: (557, 40)
2024 accident records: 557
2024 preliminary-task dataset saved to: /Users/linda/Desktop/2AMD20-Knowledge-Engineering/data/clean/ds2_2024.csv
